# Semana 7 · ¿A qué distancia está la Luna y cómo lo sabemos?

Este cuaderno acompaña a la sesión de proyecto de la semana 7: resolver triángulos rectángulos y usarlos para calcular la distancia a la Luna por tres vías distintas.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar.

Tiempo estimado: 30-45 minutos.

## 1. Seno, coseno y tangente

En un triángulo rectángulo con un ángulo agudo $\alpha$:

$$\sin\alpha = \frac{\text{opuesto}}{\text{hipotenusa}},\qquad \cos\alpha = \frac{\text{adyacente}}{\text{hipotenusa}},\qquad \tan\alpha = \frac{\text{opuesto}}{\text{adyacente}}.$$

NumPy trabaja en **radianes**, así que convertimos los grados con `np.radians`.

In [ ]:
import numpy as np

def sen(grados):  return np.sin(np.radians(grados))
def cos(grados):  return np.cos(np.radians(grados))
def tan(grados):  return np.tan(np.radians(grados))

for a in [30, 45, 60]:
    print(f"{a}°: sen = {sen(a):.4f}   cos = {cos(a):.4f}   tan = {tan(a):.4f}   sen² + cos² = {sen(a)**2 + cos(a)**2:.4f}")

Fíjate en la última columna: $\sin^2\alpha + \cos^2\alpha = 1$ siempre, que es el teorema de Pitágoras con la hipotenusa igual a 1.

**Ejercicio.** Una escalera de 5 m está apoyada formando 60° con el suelo. ¿A qué altura llega y a qué distancia de la pared está su base?

In [ ]:
# Tu código aquí

In [ ]:
# Solución
L = 5
print("altura:", L * sen(60), "m   base:", L * cos(60), "m")

## 2. Resolver un triángulo rectángulo

Si conoces dos elementos (un lado y un ángulo, o dos lados), puedes hallar los demás.

In [ ]:
def resolver(hipotenusa=None, cateto_opuesto=None, cateto_adyacente=None, angulo=None):
    """Devuelve (hipotenusa, opuesto, adyacente, ángulo) con dos datos conocidos (el ángulo en grados)."""
    if hipotenusa and angulo is not None:
        return tuple(round(float(v), 4) for v in (hipotenusa, hipotenusa * sen(angulo), hipotenusa * cos(angulo), angulo))
    if cateto_adyacente and angulo is not None:
        return tuple(round(float(v), 4) for v in (cateto_adyacente / cos(angulo), cateto_adyacente * tan(angulo), cateto_adyacente, angulo))
    if cateto_opuesto and cateto_adyacente:
        return tuple(round(float(v), 4) for v in (np.hypot(cateto_opuesto, cateto_adyacente), cateto_opuesto, cateto_adyacente, np.degrees(np.arctan2(cateto_opuesto, cateto_adyacente))))
    raise ValueError("Faltan datos")

print(resolver(hipotenusa=10, angulo=30))
print(resolver(cateto_opuesto=3, cateto_adyacente=4))

## 3. Paralaje: dos miradas, un triángulo

Si dos observadores separados una **línea de base** $B$ miran un objeto lejano, lo ven en direcciones que difieren un ángulo, la **paralaje**. En el triángulo isósceles que forman, con la base $B$ y el ángulo $p$ en el objeto, la distancia es aproximadamente $D \approx B / p$ con $p$ en radianes. Para ver cuánto vale esa aproximación, la comparamos con la solución exacta, $D = \dfrac{B/2}{\tan(p/2)}$.

In [ ]:
def distancia_paralaje(B, p_grados, exacto=True):
    """Distancia a un objeto visto con paralaje p (grados) desde dos puntos separados B."""
    if exacto:
        return (B / 2) / tan(p_grados / 2)
    return B / np.radians(p_grados)

print(distancia_paralaje(10, 0.5), distancia_paralaje(10, 0.5, exacto=False))    # base de 10 m, paralaje de 0,5°

## 4. La distancia a la Luna por paralaje

La **paralaje horizontal** de la Luna es el ángulo bajo el que se vería, desde la Luna, el radio de la Tierra. Equivale a observar la Luna desde dos puntos opuestos de la Tierra (línea de base de un diámetro) y tomar la mitad de la diferencia. En el triángulo rectángulo formado por el centro de la Tierra, el observador y la Luna,

$$\sin p = \frac{R_{\oplus}}{D} \quad\Rightarrow\quad D = \frac{R_{\oplus}}{\sin p}.$$

In [ ]:
R_tierra = 6378.137     # km, radio ecuatorial

def distancia_luna(p_grados):
    return R_tierra / sen(p_grados)

p = 57 / 60            # 57 minutos de arco, en grados: valor medio de la paralaje horizontal de la Luna
print(f"p = {p:.4f}°  ->  D = {distancia_luna(p):,.0f} km  ({distancia_luna(p) / R_tierra:.1f} radios terrestres)")

# Sensibilidad: ¿cuánto cambia D si la paralaje se mide con un error de 1′?
for dp in [-1, 0, 1]:
    print(f"paralaje {57 + dp}′ -> {distancia_luna((57 + dp) / 60):,.0f} km")

Un minuto de arco de error en la paralaje (de los 57′ que hay que medir) se traduce en unos 7000 km de error en la distancia: la medida es difícil porque el ángulo es pequeño.

## 5. La distancia a la Luna por la sombra de la Tierra

Durante un eclipse de Luna, la sombra de la Tierra en el cono de sombra tiene un tamaño que se puede comparar con el de la Luna. La longitud del cono de sombra sale de triángulos semejantes (semana 6) con los radios del Sol y de la Tierra y la distancia entre ellos.

In [ ]:
R_sol, R_tierra_media, UA = 6.957e5, 6371.0, 1.496e8     # km
L_cono = UA * R_tierra_media / (R_sol - R_tierra_media)  # longitud del cono de sombra (umbra)
D_luna = 3.844e5
radio_sombra = R_tierra_media * (1 - D_luna / L_cono)    # radio de la sombra a la distancia de la Luna
print(f"Longitud del cono de sombra: {L_cono:,.0f} km")
print(f"Diámetro de la sombra a la distancia de la Luna: {2 * radio_sombra:,.0f} km = {2 * radio_sombra / 3474:.2f} diámetros lunares")

La sombra de la Tierra en la órbita de la Luna mide unas 2,6 o 2,7 veces el diámetro lunar. Medir ese cociente en un eclipse permite deducir cuánto mide la Luna en comparación con la Tierra y, con su tamaño angular (semana 2), su distancia.

## 6. La distancia a la Luna con un láser

Los reflectores dejados en la Luna permiten medir el tiempo que tarda un pulso de luz en ir y volver. La luz viaja a $c = 299\,792{,}458$ km/s, de modo que $D = c\,t/2$.

In [ ]:
c_luz = 299792.458        # km/s
t_ida_vuelta = 2.564      # s (valor de ejemplo)
print(f"D = {c_luz * t_ida_vuelta / 2:,.0f} km")
print(f"Ida y vuelta a 384 400 km: {2 * 384400 / c_luz:.3f} s")

## 7. Para pensar

1. Si la paralaje horizontal fuese 1°, ¿a qué distancia estaría la Luna? ¿Y si fuese 0,5°?
2. ¿Por qué la distancia a las estrellas no se puede medir con una línea de base de un radio terrestre? (Pista: calcula la paralaje de una estrella a 4 años luz con esa línea de base: 1 año luz ≈ $9{,}46\times10^{12}$ km.)
3. Con el resultado de la sesión 5, calcula la precisión que necesita un láser (tiempo de ida y vuelta) para medir la distancia a la Luna con una incertidumbre de 1 cm.